In [ ]:
import numpy as np 
import xarray as xr
import pandas as pd
import ast
import xarray as xr
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


L2_merged_path = '/Volumes/Extreme SSD/PhD backup/Satellite Data/merged_datasets/modis_merged_20260717_0724.nc'


In [ ]:
# lat_min = 55
# lat_max = 65
# lon_min = -20
# lon_max = 5

sat = xr.open_dataset(L2_merged_path)
# sat = sat.sel(latitude=slice(55, 65), longitude=slice(-20, 5))

print(sat)

def make_rgb_MODIS(ds, var="Rrs", plo=2, phi=98, gamma=0.5):
    # Extract 3 bands as 2D arrays
    r = ds['Rrs_678']
    g = ds['Rrs_555']
    b = ds['Rrs_469']

    # Stack to (lat, lon, 3)
    rgb = np.stack([r, g, b], axis=-1).astype(np.float32)

    # Valid mask for percentile calc
    valid = np.isfinite(rgb).all(axis=-1)

    # Per‑channel percentile stretch
    for i in range(3):
        lo, hi = np.nanpercentile(rgb[valid, i], [plo, phi])
        rgb[:, :, i] = np.clip((rgb[:, :, i] - lo) / (hi - lo + 1e-12), 0, 1)

    # Gamma correction for brightness
    rgb = np.where(np.isfinite(rgb), np.power(rgb, gamma), np.nan)

    # Fill NaNs as white so clouds are visible
    rgb[~np.isfinite(rgb).all(axis=-1)] = 1.0

    # Final clip to [0, 1]
    return np.clip(rgb, 0, 1)


def make_rgb(ds,plo=2, var = "Rrs", phi=98, gamma=0.5):
    # Extract 3 bands as 2D arrays
    r = ds[var].sel(wavelength_3d=660.0).values
    g = ds[var].sel(wavelength_3d=550.0).values
    b = ds[var].sel(wavelength_3d=470.0).values

    # Stack to (lat, lon, 3)
    rgb = np.stack([r, g, b], axis=-1).astype(np.float32)

    # Valid mask for percentile calc
    valid = np.isfinite(rgb).all(axis=-1)

    # Per‑channel percentile stretch
    for i in range(3):
        lo, hi = np.nanpercentile(rgb[valid, i], [plo, phi])
        rgb[:, :, i] = np.clip((rgb[:, :, i] - lo) / (hi - lo + 1e-12), 0, 1)

    # Gamma correction for brightness
    rgb = np.where(np.isfinite(rgb), np.power(rgb, gamma), np.nan)

    # Fill NaNs as white so clouds are visible
    rgb[~np.isfinite(rgb).all(axis=-1)] = 1.0

    # Final clip to [0, 1]
    return np.clip(rgb, 0, 1)



rgb = make_rgb_MODIS(sat)
lons = sat['longitude']
lats = sat['latitude']

In [ ]:




# 1.1 Create RGB figure
fig = px.imshow(
    rgb,
    x=lons,           # x‑axis coordinates (columns → longitudes)
    y=lats,           # y‑axis coordinates (rows → latitudes)
    origin="lower",   # row 0 is at bottom → 50°N, not flipped
    aspect="auto",
    labels={"x": "Longitude", "y": "Latitude"},
)



# 1.2 Tidy layout (colors, margins, background)
fig.update_layout(
    template="plotly_dark",  # nice dark theme; change to 'plotly_white' if you prefer
    margin=dict(l=40, r=20, t=40, b=40),
    coloraxis_showscale=False,  # hide image colorbar (it’s RGB)
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    height=650,
)

